# P1.01 · Inferência de rejeitados (com gabarito)
**Motor de concessão de crédito E2E** · camada **L4**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1thirteeng3/ds-ai-cookbook/blob/colab/pt/projects/P1-credit/P1.01-e2e-credit-decision-engine/notebooks/40-reject-inference.ipynb)

| Nível | Runtime Colab | Tempo (smoke / full) | Documento |
|---|---|---|---|
| 🔴 L4 | cpu | 2 / 10 min | [index.pt.md](../index.pt.md) |

> Todo modelo de concessão é treinado só com quem foi aprovado no passado. Simulamos uma política antiga, escondemos o resultado dos recusados e comparamos quatro técnicas de inferência de rejeitados contra o gabarito, coisa impossível com dados reais de produção.

## 0. Parâmetros

`RUN_MODE="smoke"` usa 10% da base (ou o gêmeo sintético com `OFFLINE=True`) e roda em minutos. `RUN_MODE="full"` usa a base inteira e produz os resultados oficiais. `OFFLINE=True` só existe para testes e CI: nunca tire conclusões dele.

In [ ]:
# Parameters (papermill overrides this cell)
RUN_MODE = "smoke"   # "smoke" | "full"
SEED = 42
DATA_DIR = "data"    # Colab + Drive: "/content/drive/MyDrive/cookbook-data"
OFFLINE = False      # True: synthetic twin (tests/CI). Never for full results.
SAMPLE_FRAC = 0.1    # smoke only
N_BOOT = 200         # bootstrap resamples for metric uncertainty
REJECT_RATE = 0.30     # share declined by the simulated historical policy
PARCEL_FACTOR = 1.5    # parceling: rejects default this much more than accepts in the same band

## 1. Setup

No Colab, instala o pacote `cookbook` do repositório privado (segredo `GH_TOKEN`) e as versões exatas usadas nos resultados. NumPy, pandas e scikit-learn ficam com as versões do Colab e são registrados no arquivo de resultados. Se a instalação atualizar algum deles, use *Ambiente de execução > Reiniciar sessão* e rode a partir da seção 0. Para baixar o Home Credit, crie o segredo `KAGGLE_API_TOKEN` (ou `KAGGLE_USERNAME` e `KAGGLE_KEY`) e aceite as regras da competição no kaggle.com com a mesma conta.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = {
    "optbinning": "1.0.0",
    "lightgbm": "4.7.0",
    "optuna": "5.0.0",
    "mlflow": "3.18.0",
    "fastapi": "0.162.0",
    "httpx": "0.28.1",
    "polars": "2.1.0",
    "pyarrow": "26.0.1",
    "kaggle": "2.2.5",
    "joblib": "1.6.0",
    "matplotlib": "3.11.2",
}

if IN_COLAB and not sys.modules.get("cookbook"):
    from google.colab import userdata

    token = userdata.get("GH_TOKEN")
    pins = " ".join(f"{k}=={v}" for k, v in PACKAGES.items())
    %pip install -q uv
    !uv pip install --system -q {pins} "cookbook[credit,kaggle] @ git+https://{token}@github.com/1thirteeng3/cookbokk.git@main"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from cookbook import credit as cr
from cookbook import home_credit as hc
from cookbook.data import sample
from cookbook.env import colab_secrets_to_env, hardware_info, quiet_logs, set_seeds, versions
from cookbook.results import Timer, save_result

quiet_logs()  # before heavy imports: silences optional-solver probing
import json

import lightgbm as lgb
from sklearn.linear_model import LogisticRegression

set_seeds(SEED)
colab_secrets_to_env()
if RUN_MODE == "full" and OFFLINE:
    raise ValueError("RUN_MODE='full' needs real data (OFFLINE=False).")

SOURCE = "home-credit-synthetic" if OFFLINE else "home-credit"
ARTIFACTS = Path(DATA_DIR) / "artifacts" / "P1.01" / RUN_MODE
ARTIFACTS.mkdir(parents=True, exist_ok=True)
HW = hardware_info()
VERSIONS = versions([*PACKAGES, "numpy", "pandas", "scikit-learn"])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)
print(SOURCE, RUN_MODE, ARTIFACTS, HW)

In [ ]:
missing = [f for f in ["features.parquet", "feature_meta.json", "model_card.json"] if not (ARTIFACTS / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {ARTIFACTS}. Run notebooks 30 and 31 first "
        f"with the same RUN_MODE and DATA_DIR. / Rode 30 e 31 antes."
    )

## 2. O problema e a simulação

Quem foi recusado no passado não tem resultado observado. Treinar só com aprovados (*known good/bad*, KGB) cria **viés de seleção**: o modelo nunca viu o perfil mais arriscado e tende a subestimar o PD justamente onde a nova política vai decidir.

Home Credit só contém empréstimos concedidos, então todos têm rótulo. Isso permite um experimento controlado:

1. Uma **política histórica** (regressão logística simples sobre 3 variáveis) recusa os 30% de maior risco.
2. No treino, escondemos o rótulo dos recusados.
3. Cada técnica treina com o que "teria" disponível e é avaliada no teste **com todos os rótulos** (*through-the-door*, TTD).

In [ ]:
feats = pd.read_parquet(ARTIFACTS / "features.parquet")
meta = json.loads((ARTIFACTS / "feature_meta.json").read_text())
card = json.loads((ARTIFACTS / "model_card.json").read_text())
DATASET = meta["dataset"]
COLS = meta["model_columns"]
X, _ = hc.to_categorical(feats[COLS], meta["categories"])
y = feats[hc.TARGET].to_numpy()
part = feats["SPLIT"].to_numpy()
tr, te = part == "train", part == "test"

HIST_VARS = ["EXT_SOURCE_2", "AGE_YEARS", "CREDIT_INCOME"]
Z = feats[HIST_VARS].fillna(feats.loc[tr, HIST_VARS].median())
old_policy = LogisticRegression(max_iter=1000).fit(Z[tr], y[tr])
old_pd = old_policy.predict_proba(Z)[:, 1]
accepted = old_pd <= np.quantile(old_pd, 1 - REJECT_RATE)
pd.DataFrame({
    "share": {"accepted": accepted.mean(), "rejected": (~accepted).mean()},
    "true bad rate (hidden in practice)": {"accepted": y[accepted].mean(), "rejected": y[~accepted].mean()},
})

## 3. Solução: quatro técnicas e dois limites

Todas usam o mesmo LightGBM (hiperparâmetros do notebook 31, número fixo de árvores) e só mudam os dados de treino.

| Técnica | Como trata os recusados |
|---|---|
| **Oráculo** (limite superior) | usa o rótulo verdadeiro, indisponível na prática |
| **KGB** (limite inferior usual) | ignora os recusados |
| **Corte rígido** | recusado com PD do KGB acima do *break-even* vira mau, os demais bons |
| **Aumento *fuzzy*** | cada recusado entra duas vezes: mau com peso PD, bom com peso 1 − PD |
| **Parcelamento** | por faixa de PD do KGB, sorteia mau com a taxa da faixa × `PARCEL_FACTOR` |

In [ ]:
PARAMS = {"objective": "binary", "verbose": -1, "seed": SEED, "bagging_freq": 1, **card["best_params"]}
N_ROUNDS = max(int(card["best_iteration"]), 50)

def fit(Xtr, ytr, weight=None):
    return lgb.train(PARAMS, lgb.Dataset(Xtr, ytr, weight=weight), N_ROUNDS)

acc_tr, rej_tr = tr & accepted, tr & ~accepted
X_acc, y_acc, X_rej = X[acc_tr], y[acc_tr], X[rej_tr]
rng = np.random.default_rng(SEED)

with Timer() as t_all:
    models = {"oracle (all true labels)": fit(X[tr], y[tr]), "KGB (accepted only)": fit(X_acc, y_acc)}
    pd_rej = models["KGB (accepted only)"].predict(X_rej)

    hard = (pd_rej >= 0.10 / (0.10 + 0.60)).astype(int)
    models["hard cut-off"] = fit(pd.concat([X_acc, X_rej]), np.r_[y_acc, hard])

    models["fuzzy augmentation"] = fit(
        pd.concat([X_acc, X_rej, X_rej]), np.r_[y_acc, np.ones(len(X_rej)), np.zeros(len(X_rej))],
        weight=np.r_[np.ones(len(X_acc)), pd_rej, 1 - pd_rej],
    )

    pd_acc = models["KGB (accepted only)"].predict(X_acc)
    edges = np.unique(np.quantile(pd_acc, np.linspace(0, 1, 11)))
    band_acc, band_rej = np.digitize(pd_acc, edges[1:-1]), np.digitize(pd_rej, edges[1:-1])
    band_rate = pd.Series(y_acc).groupby(band_acc).mean()
    parcel = rng.binomial(1, np.clip(band_rate.reindex(band_rej).fillna(y_acc.mean()).to_numpy() * PARCEL_FACTOR, 0, 1))
    models["parceling"] = fit(pd.concat([X_acc, X_rej]), np.r_[y_acc, parcel])
print(f"trained {len(models)} models in {t_all.seconds:.0f}s")

## 4. Validação contra o gabarito

- **Gini TTD:** ordenação sobre todos os pedidos de teste, o que importa para a política nova.
- **Gini aprovados:** só onde a política antiga aprovava (o que se mede na prática).
- **Erro de PD nos recusados:** PD médio previsto menos inadimplência real dos recusados do teste. Negativo = subestima o risco.
- **Inadimplência a 70% de aprovação:** a política nova aprovando 70% de todos os pedidos de teste.

In [ ]:
y_te, acc_te = y[te], accepted[te]
rows = {}
for name, model in models.items():
    p = model.predict(X[te])
    approve = p <= np.quantile(p, 0.70)
    rows[name] = {
        "gini_ttd": cr.bootstrap(cr.gini, y_te, p, n=N_BOOT, seed=SEED),
        "gini_accepted": cr.gini(y_te[acc_te], p[acc_te]),
        "pd_error_rejects": p[~acc_te].mean() - y_te[~acc_te].mean(),
        "bad_rate_at_70": y_te[approve].mean(),
    }
RI = pd.DataFrame({k: {**v, "gini_ttd": v["gini_ttd"]["mean"], "gini_ttd_std": v["gini_ttd"]["std"]} for k, v in rows.items()}).T
RI

### 4.1 Como ler

O resultado típico na literatura e na prática: a inferência de rejeitados **muda pouco a ordenação** e **mexe mais na calibração** da região recusada. Às vezes piora tudo, porque as premissas (por exemplo, o `PARCEL_FACTOR`) são chutes. A distância entre KGB e oráculo é o máximo que qualquer técnica poderia recuperar. Se é pequena, a técnica não compensa a complexidade. Só uma simulação como esta mostra isso, por isso ela vem antes de adotar a técnica em produção.

## 5. Eixos de variação

### 5.1 Quanto a política antiga recusava

Quanto mais a política antiga recusava, maior o viés de seleção. Repetimos KGB e oráculo para 10%, 30% e 50% de recusa.

In [ ]:
axis = []
p_oracle = models["oracle (all true labels)"].predict(X[te])
for rate in (0.10, 0.30, 0.50):
    acc = old_pd <= np.quantile(old_pd, 1 - rate)
    kgb = fit(X[tr & acc], y[tr & acc]).predict(X[te])
    axis.append({"reject_rate": rate, "gini_ttd_kgb": cr.gini(y_te, kgb), "gini_ttd_oracle": cr.gini(y_te, p_oracle),
                 "pd_error_rejects_kgb": kgb[~acc[te]].mean() - y_te[~acc[te]].mean()})
pd.DataFrame(axis)

## 6. Armadilhas demonstradas

### 6.1 Validar só nos aprovados

**Sintoma:** a técnica "não muda nada" na validação. **Causa:** a validação usual só tem aprovados, exatamente onde todas as técnicas concordam. Compare `gini_accepted` com `gini_ttd` na tabela da seção 4. **Correção:** simulação com gabarito, ou uma pequena faixa de aprovação aleatória (*below-cut-off testing*) para obter rótulos na região recusada.

### 6.2 Tratar todos os recusados como maus

Ver a linha *hard cut-off*: com um corte agressivo, o modelo aprende a reproduzir a política antiga em vez de prever inadimplência.

## 7. Resultados

In [ ]:
for name, r in rows.items():
    save_result(
        recipe_id="P1.01", notebook="40-reject-inference", stack="lightgbm-reject-inference", variant=name,
        run_mode=RUN_MODE, hardware=HW, versions=VERSIONS, dataset=DATASET,
        params={"reject_rate": REJECT_RATE, "parcel_factor": PARCEL_FACTOR, "n_rounds": N_ROUNDS},
        metrics={"gini_ttd": r["gini_ttd"],
                 **{k: {"mean": round(float(r[k]), 6), "std": None, "n": 1}
                    for k in ("gini_accepted", "pd_error_rejects", "bad_rate_at_70")}},
        time_s={"train": round(t_all.seconds / len(models), 3), "inference_per_1k": None},
    )

## 8. Conclusão

- O viés de seleção é real e cresce com a taxa de recusa histórica.
- As técnicas clássicas recuperam, no máximo, a distância entre KGB e oráculo, e dependem de premissas não verificáveis.
- Em produção, a forma mais confiável de obter informação sobre recusados é **experimentar**: aprovar uma pequena fração aleatória abaixo do corte (receita N1.03).